In [3]:
#Этот скрипт выполнит следующие действия:
#1.  Загрузит данные о заказах и складе.
#2.  Сгруппирует все заказы по их `MpId`.
#3.  Создаст интерактивную карту, центрированную на складе.
#4.  Для каждого микрополигона:
#    *   Назначит случайный цвет.
#    *   Нанесет на карту все точки заказов в виде маленьких кругов.
#    *   Построит выпуклую оболочку (convex hull), чтобы очертить границы полигона.
#5.  Добавит на карту маркер склада.
#6.  Сохранит результат в файл `micropolygons_map.html`.

#!/usr/bin/env python3
import json
import random
from collections import defaultdict
from pathlib import Path

import folium
import numpy as np
from scipy.spatial import ConvexHull

# --- Конфигурация ---
# Укажите пути к вашим файлам данных
ORDERS_FILE = "ml_ozon_logistic_dataSetOrders.json"
COURIERS_FILE = "ml_ozon_logistic_dataSetCouriers.json"
OUTPUT_MAP_FILE = "micropolygons_map.html"

def load_json_data(path):
    """Загружает JSON данные из файла."""
    print(f"Загрузка данных из {path}...")
    return json.loads(Path(path).read_text(encoding="utf-8"))

def generate_random_color():
    """Генерирует случайный цвет в формате HEX."""
    return f"#{random.randint(0, 0xFFFFFF):06x}"

def main():
    """Основная функция для создания карты визуализации."""
    # 1. Загрузка данных
    orders_data = load_json_data(ORDERS_FILE)["Orders"]
    couriers_data = load_json_data(COURIERS_FILE)
    warehouse_info = couriers_data["Warehouse"]

    # 2. Группировка заказов по микрополигонам
    mp_orders = defaultdict(list)
    for order in orders_data:
        mp_orders[order["MpId"]].append(order)

    print(f"Найдено {len(orders_data)} заказов в {len(mp_orders)} микрополигонах.")

    # 3. Создание базовой карты
    # Центрируем карту на складе
    map_center = [warehouse_info["Lat"], warehouse_info["Long"]]
    m = folium.Map(location=map_center, zoom_start=11)

    # Добавляем маркер для склада
    folium.Marker(
        location=[warehouse_info["Lat"], warehouse_info["Long"]],
        popup=f"<strong>Склад (ID: {warehouse_info['ID']})</strong>",
        icon=folium.Icon(color="red", icon="industry", prefix="fa"),
    ).add_to(m)

    print("Создание визуализации для каждого микрополигона...")
    # 4. Итерация по микрополигонам для их отрисовки
    for mp_id, orders in mp_orders.items():
        if not orders:
            continue

        color = generate_random_color()
        points = []

        # Добавляем точки заказов на карту
        for order in orders:
            lat, lon = order["Lat"], order["Long"]
            points.append([lat, lon])
            folium.CircleMarker(
                location=[lat, lon],
                radius=3,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.7,
                popup=f"Заказ ID: {order['ID']}<br>MpId: {mp_id}",
            ).add_to(m)

        # 5. Отрисовка границ полигона (выпуклая оболочка)
        # Для построения оболочки нужно минимум 3 точки
        if len(points) >= 3:
            try:
                # Конвертируем в numpy массив для Scipy
                points_np = np.array(points)
                hull = ConvexHull(points_np)
                # Получаем вершины оболочки в правильном порядке
                hull_points = points_np[hull.vertices, :]
                
                folium.Polygon(
                    locations=hull_points.tolist(),
                    color=color,
                    weight=2,
                    fill=True,
                    fill_color=color,
                    fill_opacity=0.1,
                    tooltip=f"Микрополигон ID: {mp_id}<br>Заказов: {len(orders)}",
                ).add_to(m)
            except Exception as e:
                print(f"Не удалось построить оболочку для MpId {mp_id}: {e}")


    # 6. Сохранение карты в HTML файл
    m.save(OUTPUT_MAP_FILE)
    print(f"\nВизуализация готова! Откройте файл '{OUTPUT_MAP_FILE}' в вашем браузере.")


if __name__ == "__main__":
    main()

Загрузка данных из ml_ozon_logistic_dataSetOrders.json...
Загрузка данных из ml_ozon_logistic_dataSetCouriers.json...
Найдено 20160 заказов в 1394 микрополигонах.
Создание визуализации для каждого микрополигона...

Визуализация готова! Откройте файл 'micropolygons_map.html' в вашем браузере.
